In [ ]:
"""Shared GROMACS setup for the p16-CDK6 simulations. Runs inside a Kaggle kernel."""
import os, subprocess, sys, shutil, textwrap, time
from pathlib import Path
import urllib.request

WORK = Path('/kaggle/working')
MM = Path('/kaggle/tmp/mm')
ENV = 'md'


def sh(cmd, inp=None, check=True, tail=25, cwd=None, quiet=False):
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, input=inp, text=True, capture_output=True, cwd=cwd, executable='/bin/bash')
    out = r.stdout + r.stderr
    if not quiet or r.returncode:
        print(f'$ {cmd[:150]}  [{time.time() - t0:.0f}s, exit {r.returncode}]')
        print('\n'.join(out.splitlines()[-tail:]))
    if check and r.returncode:
        raise RuntimeError(f'failed: {cmd[:150]}')
    return out


def install():
    """Micromamba env with a CUDA build of GROMACS and the Python model-building tools."""
    MM.mkdir(parents=True, exist_ok=True)
    sh(f'cd {MM} && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba')
    print(sh(f'{MM}/bin/micromamba search -c conda-forge "gromacs>=2023" 2>&1 | tail -14', check=False, quiet=True))
    sh(f'{MM}/bin/micromamba create -y -q -r {MM}/r -n {ENV} -c conda-forge "gromacs=*=*cuda*" python=3.11 pdbfixer openmm numpy', check=False, tail=8)
    gmx = f'{MM}/r/envs/{ENV}/bin.AVX2_256/gmx'
    if not Path(gmx).exists():
        gmx = f'{MM}/r/envs/{ENV}/bin/gmx'
    return gmx


def gmx_ok(gmx):
    if not Path(gmx).exists():
        return False
    v = sh(f'{gmx} --version', check=False, quiet=True)
    print('\n'.join(l for l in v.splitlines() if any(k in l for k in ('GROMACS version', 'GPU support', 'SIMD', 'FFT library'))))
    return 'GPU support:        CUDA' in v or 'CUDA' in [l for l in v.splitlines() if 'GPU support' in l][0]


EM = """integrator=steep
emtol=1000
emstep=0.01
nsteps=50000
cutoff-scheme=Verlet
coulombtype=PME
rcoulomb=1.0
rvdw=1.0
pbc=xyz
"""
BASE = """cutoff-scheme=Verlet
coulombtype=PME
rcoulomb=1.0
vdwtype=cut-off
rvdw=1.0
DispCorr=EnerPres
pbc=xyz
tcoupl=V-rescale
tc-grps=System
tau_t=1.0
ref_t=310
constraints=h-bonds
constraint-algorithm=lincs
"""
NVT = BASE + """integrator=md
define=-DPOSRES
dt=0.002
nsteps=100000
nstenergy=5000
nstlog=5000
pcoupl=no
gen_vel=yes
gen_temp=310
gen_seed={seed}
"""
NPT = BASE + """integrator=md
define=-DPOSRES
refcoord_scaling=com
dt=0.002
nsteps=250000
nstenergy=5000
nstlog=5000
pcoupl=C-rescale
tau_p=2.0
ref_p=1.0
compressibility=4.5e-5
continuation=yes
gen_vel=no
"""
PROD = BASE + """integrator=md
dt=0.004
mass-repartition-factor=3
nsteps={nsteps}
nstxout-compressed=12500
compressed-x-precision=1000
compressed-x-grps=Protein
nstenergy=12500
nstlog=25000
pcoupl=C-rescale
tau_p=5.0
ref_p=1.0
compressibility=4.5e-5
continuation=yes
gen_vel=no
"""


In [ ]:
RUNS = ['wt-1', 'r24p-1', 'd84n-1', 'h123q-1', 'wt-2', 'r24p-2']
HOURS = 5.5
"""Production chunk: runs its replicas on two GPUs, one after the other, and stops itself before the session limit."""
import glob, threading, re
LOG = open(WORK / 'run.log', 'a', buffering=1)
def say(*a):
    m = ' '.join(str(x) for x in a)
    print(m, flush=True)
    LOG.write(time.strftime('%H:%M:%S ') + m + '\n')
T0 = time.time()
DEADLINE = T0 + HOURS * 3600
say('runs', RUNS, 'hours', HOURS)
GMX = install()
say('gmx ok', gmx_ok(GMX), GMX)
say(sh('nvidia-smi -L', quiet=True))
INP = Path('/kaggle/input')
SYSROOT = next(Path(p).parents[1] for p in glob.glob(str(INP / '**/systems/wt/topol.top'), recursive=True))
say('systems from', SYSROOT)
SYS = ['wt', 'r24p', 'd84n', 'h123q']
NSTEPS = 6250000
(WORK / 'em.mdp').write_text(EM)
(WORK / 'npt.mdp').write_text(NPT)
(WORK / 'md.mdp').write_text(PROD.format(nsteps=NSTEPS))
GPU = '-nb gpu -pme gpu -bonded gpu -update gpu -ntmpi 1'


def prev(rid):
    found = sorted(glob.glob(str(INP / f'**/runs/{rid}/md.cpt'), recursive=True), key=lambda p: int(re.search(r'senguard-msc-(\w)', p).group(1), 36) if re.search(r'senguard-msc-(\w)', p) else 0)
    return Path(found[-1]).parent if found else None


def one(rid, g, pin, maxh):
    name, rep = rid.split('-')
    d = WORK / 'runs' / rid
    d.mkdir(parents=True, exist_ok=True)
    p = prev(rid)
    tag = f'-pin on -pinoffset {pin} -pinstride 1 -ntomp 2 -gpu_id {g}'
    def run(cmd, tail=2):
        out = sh(cmd, cwd=d, check=False, quiet=True)
        LOG.write(f'[{rid}] $ {cmd[:120]}\n' + '\n'.join(l for l in out.splitlines() if re.search(r'WARNING|NOTE|Fatal|Error|Performance|error', l))[:1500] + '\n')
        return out
    if p:
        shutil.copy(p / 'md.tpr', d / 'md.tpr'); shutil.copy(p / 'md.cpt', d / 'md.cpt')
        extra = '-cpi md.cpt'
        say(rid, 'resume from', p)
    else:
        src = SYSROOT / 'systems' / name
        for f in src.glob('*'):
            if f.suffix in ('.top', '.itp') or f.name == 'ionized.gro':
                shutil.copy(f, d / f.name)
        seed = 2026 + 100 * SYS.index(name) + int(rep)
        (d / 'nvt.mdp').write_text(NVT.format(seed=seed))
        run(f'{GMX} grompp -f {WORK}/em.mdp -c ionized.gro -p topol.top -o em.tpr')
        run(f'{GMX} mdrun -deffnm em -nb gpu -ntmpi 1 {tag}')
        run(f'{GMX} grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr -maxwarn 1')
        run(f'{GMX} mdrun -deffnm nvt {GPU} {tag}')
        run(f'{GMX} grompp -f {WORK}/npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 1')
        run(f'{GMX} mdrun -deffnm npt {GPU} {tag}')
        run(f'{GMX} grompp -f {WORK}/md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr -maxwarn 1')
        extra = ''
        say(rid, 'equilibrated seed', seed, f'{(time.time() - T0) / 60:.0f} min since start')
    mh = max(0.05, (DEADLINE - time.time()) / 3600 / maxh)
    out = run(f'{GMX} mdrun -s md.tpr -deffnm md {extra} -noappend -maxh {mh:.3f} -cpt 15 {GPU} {tag}')
    last = re.findall(r'Performance:\s+([\d.]+)', out)
    say(rid, 'stopped; ns/day', last, 'maxh', round(mh, 2))
    for f in list(d.glob('#*')) + list(d.glob('*.trr')) + list(d.glob('nvt.*')) + list(d.glob('em.*')) + list(d.glob('mdout*')) + list(d.glob('ions*')):
        f.unlink(missing_ok=True)
    for f in d.glob('*_prev.cpt'):
        f.unlink(missing_ok=True)


def worker(g, pin, queue):
    for i, rid in enumerate(queue):
        try:
            one(rid, g, pin, len(queue) - i)
        except Exception as e:
            say(rid, 'FAILED', repr(e)[:300])


q0, q1 = RUNS[0::2], RUNS[1::2]
ts = [threading.Thread(target=worker, args=(0, 0, q0)), threading.Thread(target=worker, args=(1, 2, q1))]
for t in ts: t.start()
for t in ts: t.join()
for rid in RUNS:
    d = WORK / 'runs' / rid
    say(rid, 'files', sorted(f.name for f in d.glob('md*'))[:12], f'{sum(f.stat().st_size for f in d.glob("*")) / 1e6:.0f} MB')
say(f'KERNEL DONE {(time.time() - T0) / 3600:.2f} h')
LOG.close()
